In [5]:
import time
import sympy

# DFS and Memory

## Depth First Search (DFS) Summary

So far, we have seen (see Week 5 day 0) the technique of depth first search for searching simple graphs. The method involves starting at a source vertex and continually visiting an adjacent unvisited vertices. When there are no unvisited adjacent vertices, we backtrack until we can explore unvisited vertices. We must remember the chain of vertices through which to backrack (the string of Theseus). It is natural to store this information as a stack, since these vertices are accessed in a "last in, first out" (LIFO) pattern.

## Memory Advantage of Depth First Search over Breadth First Search

We also saw Breadth First Search (BFS) as another strategy to search a graph. Recall that in Breadth First Search, we visit the vertices in order of their distance from the source by maintaining a queue that contains all of the vertices that are unvisited but are adjacent to a vertex that has been visited. This was called the "frontier."

We saw that both strategies have cost $\Theta(|V|+|E|)$. Despite both strategies having the same cost in terms of runtime, Depth First Search does have an efficiency advantage over depth first search. There are several computational models for the searching, and this efficiency benefit depends on which computational model we employ.

### Memory as a Resource

So far, we have studied the cost of algorithms in terms of the number of operations that are needed to carry out the algorithm. However, this is just one way to measure the cost of an algorithm. The real world is complicated, and there are many possible measures of efficiency of algorithms. For example, we could measure the amount of energy (in Joules) needed to run the algorithm.

One important measure of algorithm performance is memory. Memory is the ability to store and retrieve information. Modern computers come equipped with a limited amount of Random Access Memory (RAM). If an algorithm uses more memory than is available in RAM, then it must transfer some of the information from memory into storage (i.e. the harddrive.) This is computationally expensive. Repeatedly overdrawing memory can result in a phenomenon known as [thrashing](https://en.wikipedia.org/wiki/Thrashing_(computer_science)). Therefore, we sometimes measure the performance of algorithms based on the amount of memory that they require rather than the number of operations.

Memory is especially valuable now, due to the scarcity. See the [global RAM shortage](https://en.wikipedia.org/wiki/2024%E2%80%93present_global_memory_supply_shortage).

#### Memory and Python

Memory usage is an important consideration in any programming language, including Python. Practical performance problems is Python often arise due to irresponsible memory usage.

Many programming languages (like C) require the programmer to manage memory themselves by allocating memory when needed and freeing it when it is no longer needed. This can be very tedious. For example, in programming languages like C, arrays must have fixed sizes that are determined at compile time. This makes programming a pain. C programmers must be very careful that they are not unintentionally allocating more and more by accident, known as a memory leak.

In contrast, Python makes memory management easy because the Python interpreter uses a [garbage collector](https://www.geeksforgeeks.org/python/garbage-collection-python/). Each Python variable contains a count of the number of references to it. If the count drops to zero, then the Python interpreter deletes the variable.

Improper memory usage in Python can cause problems. Consider the following example.

In [14]:
#Warning: This code block takes a long time to run. Between 1 and 2 minutes on my machine.
start = time.time()
count = 0
for n in [n for n in range(350_000_000)]: #The list is created all at once and must be held in memory.
    if n%7==1:
        count+=1
end = time.time()
print("total time for list", end-start) #takes a long time. About 53 second on my machine.

start = time.time()
count = 0
for n in (n for n in range(350_000_000)): #The generator object creates elements as they are used.
                                          #This is equivalent to the usual looping construct: for n in range(350_000_000):
    if n%7==1:
        count+=1
end = time.time()
print("total time for generator", end-start) #faster. About 24 seconds on my machine.

total time for list 52.83515524864197
total time for generator 24.84602999687195


The example above shows that when inputs are very large, memory waste can become time waste.

Note: Your machine may have different performance from mine. But there should be a size for which the second loop outperforms the first due to the need to manage memory.

When the inputs are smaller, the list is actually faster than the generator, because the list, once created, can be looped through efficiently. The drawback is that the list must be stored in memory, whereas the generator are generated when needed. When the list becomes massive, Python requires extra timesteps to manage it.

##### Iterators

Python uses an intricate web of similarly-named concepts to implement the dynamically-generated loop in the previous example. Here is a summary of the concepts, how they relate, and some of their idiosyncracies. In Lab 4, we give detailed examples.

|term | description| creation | gotchas |
|-----|------------|-----|---------|
|iterable| An iterable is an object that you can loop through.| The function ```.__iter__()``` must be defined.| Iterables are not necessarily iterators. See below.|
|iterator| An iterator is an object that is returned by an ```.__iter__()``` method| An iterator must have ```.__iter__()``` (which returns ```self```) and ```.__next__()``` defined.|Every iterator is also an iterable, but not the reverse.|
|generator function | A generator function is a function that returns a generator object| A generator function looks like a function, except that it uses the word ```yield``` instead of ```return```. | Calling a generator function returns a generator object. See below. The word "generator" alone usually refers to a generator object.|
|generator object | A generator object is an object with ```.__next__()``` defined. It is a type of iterator| Generator objects can be returned by generator functions, or by using the parenthesis-comprehension syntax, e.g. ```(x for x in l)```.| Generator objects get used up as you iterate through them. Generator objects are iterators, but not necessarily the reverse.|
|```.__iter__()```| This dunder method marks an object as an iterable.| Define it in the class, like any other dunder method.| ```.__iter__()``` can either return an iterator (often  ```self``` if self has ```.__next__()``` defined and so ```self``` is an iterator) or yield a value, in which case ```.__iter__()``` is a generator function, which, when called, returns a generator object which is a type of iterator.|
|```.__next__()```| This dunder method dictates how to extract the next element during iteration.| Define ```.__next__()``` like any other dunder method. | ```.__next__()``` should be a function, not a generator function. Use ```return``` not ```yield```. Also, typically you want to call ```next(my_object)```, not ```my_object.__next__()```.|


### Models of of computation

We want to know about the differences in memory usage between DFS vs BFS. The question is complicated, and the answer can depend on how the graph is represented and assumptions about how we measure memory consumption.

Recall our usual strategy for measuring the cost of an algorithm. We never analyze the algorithm on a particular instance of a problem. Instead, we analyze the growth behavior of a sequence of inputs of increasing size. We have a sequence of graphs $G_0,G_1,G_2,\dots$ with increasing numbers of vertices. We run BFS and DFS on each of them and compare the growth rates of the memory usages for the programs.

For DFS, we need to remember:
 - the current vertex,
 - the vertices that have been visited, 
 - the "string of Theseus" stack. (could be long)

For BFS, we need to remember: 
 - the current vertex,
 - the frontier queue,
 - the set of vertices that have been visited.

 The frontier queue will typically be larger than the string of Thesus stack, so BFS will typically require more memory than DFS, but some caveats apply.

##### Caveat 1: The graph is explicit

Suppose that the graph is given explicitly as an adjacency list, as we have seen in several illustrative examples. In this case, we need to store the entire graph in memory. If the graph is huge, then this model is only theoretical, because practical machines have limited memory.

 Since we are already storing the entire graph in memory, it takes comparatively little extra memory to store the information for BFS and DFS. If you are storing a large graph in memory, then apparently you have tons of memory and can easily afford to remember the frontier, visited set, and current vertex since these items together require less memory than the entire graph.

 In terms of asymptotics, both DFS and BFS require $\Theta(|G|)$ bits of memory, where $|G|$ is the number of bits needed to store $G$.

 Typically, we assume that the graph is stored implicitly. For example, we might be given a vertex and a function that expects a vertex returns the neighbors of the vertex. We will see an example of this in Lab 5.

 ##### Caveat 2: The search runs for a fixed number of steps and the graphs have bounded degree.

 Suppose that we only run DFS search for $k$ steps. In this case, the string of Thesus will be length at most $k$, less if backtracking has occurred. 
 
 The size of the frontier will depend on the shape of the graph. We can estimate the size by assuming that it is regular of degree $d$ (each vertex has $d$ neighbors). In this case, after $k$ steps of BFS, we will have visited $k$ vertices, and the frontier will have size at most $dk$.

This means that BFS will require at most $d$ times the amount of memory as DFS. If $d$ is fixed as the input graphs grow, then this is acceptable (memory requirements of DFS and BFS are asymptotically equal), but otherwise, BFS requires asymptotically more memory than DFS.

##### Caveat 3: The graph is a tree

If the graph is a tree and we apply depth first search from the root, then we do not need to keep track of a visited set. We continue to remember the String of Thesus stack for backtracking. We must also remember which options have been tried for each node in the String of Thesus.

In the myth of Thesus, Thesus could remember which options had been tried because they were physically arranged. After backtracking, he could decide to always take the next available path just to the left of the path he came from.

Graphs are abstract, and so they do not come with an orientation to help you decide consistently which path to try next. One possibility is to use more memory to keep track of the paths that have been tried. This is still typically a great saving over having to remember the entire visited set. In particular, it requires $\Theta(\log(d|S|))$ bits of memory, where $|S|$ is the string of Thesus stack. When $d$ is bounded, this is $\Theta(\log(d|S|))$ bits.

In most cases, we don't need to expend the extra memory to remember which of the $2^d$ of subsets of paths have been tried. For example, if each node in the tree has a name, then we can try paths in alphabetical order. This requires $\Theta(\log(\log(d)|S|))$ bits, because, for each node of $|S|$, we only need to remember which of the $d$ paths was visited last. 

##### The graph is a tree and we parameterize based on the level of tree searched.

When the graph is a tree, we are sometimes interested in knowing the amount of memory needed to search up to the $i^{th}$ level.

In order to apply breadth first search to the level $i$ of a binary tree, we need to remember $2^i$ vertices, which is the size of the frontier just before visiting the $i^{th}$ level.

Depth first search requires us to remember only $i$ vertices.

[<i>Iterative deepening</i>](https://stackoverflow.com/questions/7395992/iterative-deepening-vs-depth-first-search) is a technique that achieves the low-memory usage of depth first search with a breadth first pattern of visiting the vertices.

Iterative deepening applies depth first search from the root of a tree up, but restricts the search to nodes at level at most $i$, where initially $i=0$. Then, the algorithm increments $i$ and re-starts the search from the root. Typically, we continue until the algorithm runs out of computing power or we find our target.

Iterative deepening is an example of a <i>space/time tradeoff</i>. Iterative deepening achieves the same goal as breadth first search, yet uses less memory. It therefore requires more time. 


##### Analysis of iterative deepening

Iterative deepening requires remembering only $i$ vertices, which requires approximately $\Theta(\log(i))$ bits of memory. The tradeoff is that some vertices are visited many times, because the algorithm restarts depth first search from the beginning for each $i$. 

Suppose a binary tree has height $h$. The root will be visited $h$ times. The two children of the root will be visited $h-1$ times. Continuing this pattern, we derive the summation for the cost of iterative deepening.

$\sum_{i=0}^{h}2^{h-i} 2^i = (h+1)2^h$.

Recall that the cost of breadth first search is $\Theta(|V|+|E|)$. For our binary tree of height $h$, this is $\Theta(2^{h+1})=\Theta(2^h)$. The limit theorem demonstrates that iterative deepening is asymptotically slower than breadth first search, since 

\begin{align*}
\lim_{h\to \infty }\frac{(h+1)2^h}{2^h}=\lim_{h\to\infty}h+1 = \infty.
\end{align*}

On the other hand, iterative deepening uses dramatically less memory than depth first search. Iterative deepening uses $\Theta(\log(\log(h)))$ bits of memory. Breadth first search uses $\Theta(\log(h))$ bits of memory. If the exponential increase in memory cost of breadth first search causes the program to crash, then the speed improvement is worthless.